**Phase 1: Load & Normalize Data**

Load master table (56 students)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW students
USING CSV
OPTIONS (
  path "/Volumes/workspace/default/day1/Master_table.csv",
  header "true",
  inferSchema "true"
)

Load Task1 Responses (51 records)

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW task1_responses
USING CSV
OPTIONS (
  path "/Volumes/workspace/default/day1/Task1.csv",
  header "true",
  inferSchema "true"
)

Load Task1 File2 (60 records - has duplicates + invalid + extra)

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW task1_file2
USING CSV
OPTIONS (
  path "/Volumes/workspace/default/day1/Task 2.csv",
  header "true",
  inferSchema "true"
)


Normalize master table: lowercase/trim emails, alias columns

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW master AS
SELECT
    `REGNO`                                    AS student_id,
    `STUDENT NAME`                             AS student_name,
    LOWER(TRIM(`College Domain MAILID`))       AS college_email,
    LOWER(TRIM(`Personal Mail ID`))            AS personal_email
FROM students
WHERE `REGNO` IS NOT NULL

In [0]:

%sql
SELECT * FROM master LIMIT 5

student_id,student_name,college_email,personal_email
22PA1A4505,ALLADI JOHNPAUL,22pa1a4505@vishnu.edu.in,johnalladi0306@gmail.com
22PA1A4506,ALPESH JAIN,22pa1a4506@vishnu.edu.in,alpeshjain3112@gmail.com
22PA1A4573,MUNAGALA SAI MANIKANTA YASHWANTH,22pa1a4573@vishnu.edu.in,myaswanth18@gmail.com
22PA1A4580,NANNABATTUNI HARSHAVARDHAN CHARI,22pa1a4580@vishnu.edu.in,harshavardhanchari0304@gmail.com
22PA1A4582,NARRA LAKSHMI HANUMA,22pa1a4582@vishnu.edu.in,lakshmihanumanarra816@gmail.com


Normalize Task1 Responses: extract only valid email rows, clean timestamp

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW responses_clean AS
SELECT
    TRY_TO_TIMESTAMP(`Timestamp`, 'M-d-yyyy H:mm:ss') AS submission_time,
    LOWER(TRIM(`Email address`))                        AS email,
    TRIM(`Have you completed all the 5 queries? `)      AS completed_queries,
    TRIM(`  GitHub Username  `)                         AS github_username,
    TRIM(`Repository Link `)                            AS repo_link
FROM task1_responses
WHERE `Email address` IS NOT NULL
  AND TRIM(`Email address`) != ''
  AND LOWER(TRIM(`Email address`)) LIKE '%@%'

Normalize Task1 File2: same cleaning, different timestamp format

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW file2_clean AS
SELECT
    TRY_TO_TIMESTAMP(`Timestamp`, 'dd-MM-yyyy HH:mm') AS submission_time,
    LOWER(TRIM(`Email address`))                       AS email,
    TRIM(`Have you completed all the 5 queries? `)     AS completed_queries,
    TRIM(`  GitHub Username  `)                        AS github_username,
    TRIM(`Repository Link `)                           AS repo_link
FROM task1_file2
WHERE `Email address` IS NOT NULL
  AND TRIM(`Email address`) != ''
  AND LOWER(TRIM(`Email address`)) LIKE '%@%'

Unified email mapping — both emails → student_id

In [0]:


%sql
CREATE OR REPLACE TEMP VIEW email_mapping AS
SELECT student_id, college_email AS email FROM master WHERE college_email IS NOT NULL
UNION
SELECT student_id, personal_email AS email FROM master WHERE personal_email IS NOT NULL

In [0]:
%sql
SELECT * FROM email_mapping;


student_id,email
22PA1A0596,22pa1a0596@vishnu.edu.in
22PA1A05F6,22pa1a05f6@vishnu.edu.in
23PA5A1209,23pa5a1209@vishnu.edu.in
22PA1A05D4,22pa1a05d4@vishnu.edu.in
22PA1A4582,22pa1a4582@vishnu.edu.in
22PA1A45A1,22pa1a45a1@vishnu.edu.in
22PA1A4241,22pa1a4241@vishnu.edu.in
22PA1A42B2,22pa1a42b2@vishnu.edu.in
22PA1A4215,22pa1a4215@vishnu.edu.in
22PA1A0467,22pa1a0467@vishnu.edu.in


**Phase 2: Combine Both Submission Files**

UNION both files to get all raw submissions (before dedup/validation)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW all_submissions_raw AS
SELECT *, 'file1' AS source FROM responses_clean
UNION ALL
SELECT *, 'file2' AS source FROM file2_clean


VALID submissions

In [0]:
    
%sql
CREATE OR REPLACE TEMP VIEW valid_submissions AS
SELECT
    em.student_id,
    s.email,
    s.submission_time,
    s.completed_queries,
    s.github_username,
    s.repo_link,
    s.source
FROM all_submissions_raw s
INNER JOIN email_mapping em
ON s.email = em.email

In [0]:
%sql
SELECT COUNT(*) AS valid_count FROM valid_submissions

valid_count
107


INVALID submissions

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW invalid_submissions AS
SELECT s.email, s.submission_time, s.github_username, s.source
FROM all_submissions_raw s
LEFT ANTI JOIN email_mapping em
ON s.email = em.email


In [0]:

%sql
SELECT * FROM invalid_submissions

email,submission_time,github_username,source
varmagadiraju0318@gmail.com,2026-04-02T22:34:00.000Z,null,file2
22pa1a4244@vishnu.edu.in,2026-04-03T12:35:00.000Z,null,file2
22pa1a42b4@vishnu.edu.in,2026-04-04T13:36:00.000Z,Kasumohan29,file2
22pa1a4235@vishnu.edu.in,2026-04-05T18:39:00.000Z,null,file2


NOT SUBMITTED — students in master with no matching submission

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW not_submitted AS
SELECT m.student_id, m.student_name, m.college_email
FROM master m
LEFT ANTI JOIN valid_submissions vs
ON m.student_id = vs.student_id


In [0]:

%sql
SELECT * FROM not_submitted ORDER BY student_id

student_id,student_name,college_email


In [0]:
%sql
SELECT COUNT(*) AS not_submitted_count FROM not_submitted

not_submitted_count
0


Phase 3: Duplicate Detection

ROW_NUMBER() — rank submissions per student

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW deduped AS
SELECT *,
    ROW_NUMBER() OVER (
        PARTITION BY student_id
        ORDER BY submission_time ASC
    ) AS rn
FROM valid_submissions

First (unique) submissions per student

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW unique_submissions AS
SELECT * FROM deduped WHERE rn = 1

Duplicate submissions (same student submitted more than once)

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW duplicates AS
SELECT * FROM deduped WHERE rn > 1

In [0]:

%sql
SELECT student_id, email, submission_time, rn FROM deduped ORDER BY student_id, rn;

student_id,email,submission_time,rn
22PA1A0405,22pa1a0405@vishnu.edu.in,2026-03-30T00:35:00.000Z,1
22PA1A0405,22pa1a0405@vishnu.edu.in,2026-03-30T00:35:34.000Z,2
22PA1A0467,22pa1a0467@vishnu.edu.in,2026-03-29T20:35:00.000Z,1
22PA1A0467,22pa1a0467@vishnu.edu.in,2026-03-29T20:35:39.000Z,2
22PA1A0475,22pa1a0475@vishnu.edu.in,2026-03-29T21:22:00.000Z,1
22PA1A0475,22pa1a0475@vishnu.edu.in,2026-03-29T21:22:05.000Z,2
22PA1A04A7,22pa1a04a7@vishnu.edu.in,2026-03-29T18:09:37.000Z,1
22PA1A04A7,22pa1a04a7@vishnu.edu.in,2026-04-05T19:39:00.000Z,2
22PA1A04E0,22pa1a04e0@vishnu.edu.in,2026-03-29T22:48:00.000Z,1
22PA1A04E0,22pa1a04e0@vishnu.edu.in,2026-03-29T22:48:31.000Z,2


Compare

In [0]:

%sql
SELECT student_id, COUNT(*) AS total_submissions
FROM valid_submissions
GROUP BY student_id
HAVING COUNT(*) > 1

student_id,total_submissions
22PA1A42A9,2
22PA1A12E7,2
22PA1A12J4,2
22PA1A05F6,2
22PA1A42B2,2
22PA1A0467,2
22PA1A4580,2
22PA1A5713,2
22PA1A05D4,2
22PA1A12H4,2


**Phase 4: Advanced Insights**

Submission count per student

In [0]:
%sql
SELECT student_id, COUNT(*) AS total_submissions
FROM valid_submissions
GROUP BY student_id
ORDER BY total_submissions DESC


student_id,total_submissions
22PA1A5740,3
22PA1A42A9,2
22PA1A12E7,2
22PA1A12J4,2
22PA1A05F6,2
22PA1A42B2,2
22PA1A0467,2
22PA1A4580,2
22PA1A5713,2
22PA1A05D4,2


Students who submitted using BOTH emails (college + personal)

In [0]:

%sql
SELECT student_id, COUNT(DISTINCT email) AS emails_used
FROM valid_submissions
GROUP BY student_id
HAVING COUNT(DISTINCT email) > 1

student_id,emails_used
22PA1A5740,2


Final classification — Submitted / Duplicate / Not Submitted / Invalid

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW final_classification AS

-- Valid first-time submitters
SELECT student_id, email, submission_time, completed_queries, github_username, repo_link,
       'Submitted'     AS status
FROM unique_submissions

UNION ALL

-- Students who submitted more than once (keep the extras marked as duplicate)
SELECT student_id, email, submission_time, completed_queries, github_username, repo_link,
       'Duplicate'     AS status
FROM duplicates

UNION ALL

-- Students who never submitted
SELECT student_id, NULL AS email, NULL AS submission_time,
       NULL AS completed_queries, NULL AS github_username, NULL AS repo_link,
       'Not Submitted'  AS status
FROM not_submitted


Final dashboard — every student classified

In [0]:

%sql
SELECT * FROM final_classification ORDER BY status, student_id

student_id,email,submission_time,completed_queries,github_username,repo_link,status
22PA1A0405,22pa1a0405@vishnu.edu.in,2026-03-30T00:35:34.000Z,Yes,CharanReddyAlavala,https://github.com/CharanReddyAlavala/capgemini-data-engineering-training,Duplicate
22PA1A0467,22pa1a0467@vishnu.edu.in,2026-03-29T20:35:39.000Z,null,null,null,Duplicate
22PA1A0475,22pa1a0475@vishnu.edu.in,2026-03-29T21:22:05.000Z,Yes,22pa1a0475,https://github.com/22pa1a0475/capgemini-data-engineering-training,Duplicate
22PA1A04A7,22pa1a04a7@vishnu.edu.in,2026-04-05T19:39:00.000Z,Yes,Anusha195,https://github.com/Anusha195/capgemini-data-engineering-training,Duplicate
22PA1A04E0,22pa1a04e0@vishnu.edu.in,2026-03-29T22:48:31.000Z,Yes,kavyasamyuktha,https://github.com/kavyasamyuktha/Capgemini-Data-Engineering-Training/tree/main,Duplicate
22PA1A0551,22pa1a0551@vishnu.edu.in,2026-03-28T15:47:45.000Z,null,null,null,Duplicate
22PA1A0554,22pa1a0554@vishnu.edu.in,2026-03-28T12:55:54.000Z,Yes,deepikagundu2622,https://github.com/deepikagundu2622/capgemini-data-engineering-training.git,Duplicate
22PA1A0596,22pa1a0596@vishnu.edu.in,2026-04-05T18:24:00.000Z,Yes,Mamilla-Praveen,https://github.com/Mamilla-Praveen/Capgemini-data-engineering-training/tree/main/Week0,Duplicate
22PA1A05C0,22pa1a05c0@vishnu.edu.in,2026-03-28T18:10:08.000Z,Yes,bhavyasri9,https://github.com/bhavyasri9/capgemini-data-engineering-training,Duplicate
22PA1A05D4,22pa1a05d4@vishnu.edu.in,2026-03-28T19:49:08.000Z,Yes,22pa1a05d4,https://github.com/22pa1a05d4/capgemini-data-engineering-training/tree/main/Week0,Duplicate


Summary counts

In [0]:

%sql
SELECT status, COUNT(*) AS count
FROM final_classification
GROUP BY status
ORDER BY count DESC

status,count
Submitted,57
Duplicate,50


Invalid submissions

In [0]:
%sql
SELECT email, submission_time, github_username, source, 'Invalid' AS status
FROM invalid_submissions
ORDER BY email

email,submission_time,github_username,source,status
22pa1a4235@vishnu.edu.in,2026-04-05T18:39:00.000Z,null,file2,Invalid
22pa1a4244@vishnu.edu.in,2026-04-03T12:35:00.000Z,null,file2,Invalid
22pa1a42b4@vishnu.edu.in,2026-04-04T13:36:00.000Z,Kasumohan29,file2,Invalid
varmagadiraju0318@gmail.com,2026-04-02T22:34:00.000Z,null,file2,Invalid
